In [1]:
# https://www.matecdev.com/posts/landsat-sentinel-aws-s3-python.html
from pystac_client import Client
from json import load
import requests
from pyproj import Transformer
import rasterio as rio
import matplotlib.pyplot as plt
import satsearch
from bs4 import BeautifulSoup
from rasterio.features import bounds
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
import os
import config

In [2]:
LandsatSTAC = Client.open("https://landsatlook.usgs.gov/stac-server", headers=[])

for collection in LandsatSTAC.get_collections():
    print(collection)

<CollectionClient id=landsat-c2l2-sr>
<CollectionClient id=landsat-c2l2-st>
<CollectionClient id=landsat-c2ard-st>
<CollectionClient id=landsat-c2l2alb-bt>
<CollectionClient id=landsat-c2l3-fsca>
<CollectionClient id=landsat-c2ard-bt>
<CollectionClient id=landsat-c2l1>
<CollectionClient id=landsat-c2l3-ba>
<CollectionClient id=landsat-c2l2alb-st>
<CollectionClient id=landsat-c2ard-sr>
<CollectionClient id=landsat-c2l2alb-sr>
<CollectionClient id=landsat-c2l2alb-ta>
<CollectionClient id=landsat-c2l3-dswe>
<CollectionClient id=landsat-c2ard-ta>


In [3]:
def BuildSquare(lon, lat, delta):
    c1 = [lon + delta, lat + delta]
    c2 = [lon + delta, lat - delta]
    c3 = [lon - delta, lat - delta]
    c4 = [lon - delta, lat + delta]
    geometry = {"type": "Polygon", "coordinates": [[ c1, c2, c3, c4, c1 ]]}
    return geometry

geometry = BuildSquare(-59.346271, -34.233076, 0.04)
timeRange = '2019-06-01/2021-06-01'

In [4]:
LandsatSearch = LandsatSTAC.search ( 
    intersects = geometry,
    datetime = timeRange,
    query =  ['eo:cloud_cover95'],
    collections = ["landsat-c2l2-sr"] )

Landsat_items = [i.to_dict() for i in LandsatSearch.items()]
print(f"{len(Landsat_items)} Landsat scenes fetched")

193 Landsat scenes fetched


In [5]:

print(Landsat_items[0]['assets']['red']['href'])    
print(Landsat_items[0]['assets']['red']['alternate']['s3']['href'])

https://landsatlook.usgs.gov/data/collection02/level-2/standard/oli-tirs/2021/225/084/LC08_L2SP_225084_20210528_20210607_02_T1/LC08_L2SP_225084_20210528_20210607_02_T1_SR_B4.TIF
s3://usgs-landsat/collection02/level-2/standard/oli-tirs/2021/225/084/LC08_L2SP_225084_20210528_20210607_02_T1/LC08_L2SP_225084_20210528_20210607_02_T1_SR_B4.TIF


In [6]:
from pyproj import Transformer

def getSubset(geotiff_file, bbox):
    with rio.open(geotiff_file) as geo_fp:
        # Calculate pixels with PyProj
        Transf = Transformer.from_crs("epsg:4326", geo_fp.crs)
        lat_north, lon_west = Transf.transform(bbox[3], bbox[0])
        lat_south, lon_east = Transf.transform(bbox[1], bbox[2])
        x_top, y_top = geo_fp.index(lat_north, lon_west)
        x_bottom, y_bottom = geo_fp.index(lat_south, lon_east)

        window = rio.windows.Window.from_slices((x_top, x_bottom), (y_top, y_bottom))
        
        subset = geo_fp.read(1, window=window)
    
    return subset


In [7]:
def plotNDVI(nir,red,filename):
    ndvi = (nir-red)/(nir+red)
    ndvi[ndvi>1] = 1
    plt.imshow(ndvi)
    plt.savefig(filename)
    plt.close()

In [8]:
def download_landsat(session, url, save_path):
    response = session.get(url, stream=True)

    if response.status_code == 200:
        with open(save_path, 'wb') as f:
            for chunk in response.iter_content(1024):
                f.write(chunk)
        print(f"File downloaded successfully: {save_path}")
    else:
        print(f"Failed to download. Status code: {response.status_code}")
        print(response.text)

In [ ]:
login_url = "https://ers.cr.usgs.gov/login"
landsat_url = "https://landsatlook.usgs.gov/data/collection02/level-2/standard/oli-tirs/2021/225/084/LC08_L2SP_225084_20210528_20210607_02_T1/LC08_L2SP_225084_20210528_20210607_02_T1_SR_B4.TIF"

session = requests.Session()

response = session.get(login_url)
if response.status_code != 200:
    print(f"Failed to load login page: {response.status_code}")
    exit()

soup = BeautifulSoup(response.text, "html.parser")
csrf_token = soup.find("input", {"name": "csrf"})["value"]

username = config.USGS_USERNAME
password = config.USGS_PASSWORD 

login_payload = {
    "username": username,
    "password": password,
    "csrf": csrf_token
}

login_response = session.post(login_url, data=login_payload)

if login_response.status_code == 200:
    print("Login successful!")
else:
    print(f"Login failed: {login_response.status_code}")
    print(login_response.text)
    exit()

bbox = bounds(geometry)

for i,item in enumerate(Landsat_items):
    red_href = item['assets']['red']['href']
    nir_href =  item['assets']['nir08']['href']
    date = item['properties']['datetime'][0:10]
    red_path = os.path.join("./landsat/", f"{date}_red.tif")
    nir_path = os.path.join("./landsat/", f"{date}_nir.tif")
    download_landsat(session, red_href, red_path)
    download_landsat(session, nir_href, nir_path)

    print("Landsat item number " + str(i) + "/" + str(len(Landsat_items)) + " " + date)
    red = getSubset(red_path, bbox)
    nir = getSubset(nir_path, bbox)
    plotNDVI(nir,red,"landsat/" + date + "_ndvi.png")
    break


Login successful!
File downloaded successfully: ./landsat/2021-05-28_red.tif
File downloaded successfully: ./landsat/2021-05-28_nir.tif
Landsat item number 0/193 2021-05-28
